# PyGeoFetch — 06: Real-World Workflows

End-to-end workflows combining data acquisition, spectral indices, SAR processing,
orbit files, and processing pipelines.

### Workflows
1. NDVI time series (vegetation monitoring)
2. SAR flood mapping with Sentinel-1C
3. Burn severity mapping (dNBR)
4. Orbit file pre-download for InSAR
5. Automated weekly monitoring pipeline

In [1]:
from pygeofetch import PyGeoFetch
from pygeofetch.models.search_query import SearchQuery, BoundingBox
from pygeofetch.models.download_task import DownloadOptions
from pathlib import Path
import json

client = PyGeoFetch(log_level="INFO")
Path("output").mkdir(exist_ok=True)

20:36:18 INFO [    engine] PyGeoFetch ready


## Workflow 1: NDVI Time Series

In [2]:
# Iowa corn belt — monthly NDVI monitoring
IOWA_BBOX = "-93.8,41.9,-93.5,42.1"

query = SearchQuery(
    bbox=BoundingBox.from_string(IOWA_BBOX),
    start_date="2024-04-01",
    end_date="2024-09-30",
    cloud_cover_max=20,
    max_results=100,
    sort_by="cloud_cover",
    sort_ascending=True,
)

results = client.search(query, providers=["aws_earth"])
print(f"Found {len(results)} scenes over growing season")

# Group by month
from collections import defaultdict
monthly = defaultdict(list)
for r in results:
    if r.datetime:
        month = str(r.datetime)[:7]
        monthly[month].append(r)

print()
print(f"{'Month':<10} {'Scenes':<8} {'Best cloud%'}")
print("-" * 35)
for month in sorted(monthly):
    scenes = monthly[month]
    best_cloud = min(s.cloud_cover for s in scenes if s.cloud_cover is not None)
    print(f"  {month}   {len(scenes):<8} {best_cloud:.1f}%")

20:37:14 INFO [  searcher] Searching 1 provider(s): aws_earth
20:37:17 INFO [  searcher]   aws_earth: 71 results
20:37:17 INFO [  searcher] Search complete: 71 unique results from 1 providers
Found 71 scenes over growing season

Month      Scenes   Best cloud%
-----------------------------------
  2024-04   12       0.0%
  2024-05   12       0.2%
  2024-06   12       0.0%
  2024-07   12       4.1%
  2024-08   13       0.0%
  2024-09   10       0.0%


In [3]:
# Build the NDVI time series using the processing engine
try:
    import numpy as np
    import tempfile, rasterio
    from rasterio.transform import from_bounds
    from rasterio.crs import CRS

    ndvi_results = []
    print("Simulating NDVI computation per month (replace with real downloads):")
    print()

    for month in sorted(monthly)[:6]:
        # In production: download B04 + B08, then compute NDVI
        # result = client.indices.ndvi(red="B04.tif", nir="B08.tif")

        # Simulate growing-season NDVI curve (Iowa corn)
        month_num = int(month[5:7])
        mean_ndvi = max(0.05, 0.72 * np.exp(-0.5 * ((month_num - 7.5) / 1.8) ** 2))
        std_ndvi  = 0.05 + 0.02 * abs(month_num - 7.5)
        ndvi_results.append((month, float(mean_ndvi), float(std_ndvi)))
        print(f"  {month}  NDVI mean={mean_ndvi:.3f}  std={std_ndvi:.3f}")

except ImportError:
    print("numpy not installed — pip install numpy")

Simulating NDVI computation per month (replace with real downloads):

  2024-04  NDVI mean=0.109  std=0.120
  2024-05  NDVI mean=0.274  std=0.100
  2024-06  NDVI mean=0.509  std=0.080
  2024-07  NDVI mean=0.693  std=0.060
  2024-08  NDVI mean=0.693  std=0.060
  2024-09  NDVI mean=0.509  std=0.080


## Workflow 2: SAR Flood Mapping — Sentinel-1C

In [5]:
# Sentinel-1C is the active constellation (May 2025+)
# Same GRD format, same polarisations (VV, VH), same processing

from pygeofetch.utils.geo_utils import _normalise_satellite_name

# Demonstrate SAR fields on SatelliteData
from pygeofetch.models.satellite_data import SatelliteData

# Simulate a Sentinel-1C result
s1c_scene = SatelliteData(
    id="S1C_IW_GRDH_1SDV_20260601T053000_20260601T053027",
    provider="copernicus",
    satellite="SENTINEL-1C",
    product_type="GRD",
    polarisation="VV+VH",
    pass_direction="ascending",
    relative_orbit=37,
    gsd_m=10.0,
    incidence_angle=38.5,
)

print("Sentinel-1C scene:")
print(f"  satellite:      {s1c_scene.satellite}")
print(f"  product_type:   {s1c_scene.product_type}")
print(f"  polarisation:   {s1c_scene.polarisation}")
print(f"  pass_direction: {s1c_scene.pass_direction}")
print(f"  relative_orbit: {s1c_scene.relative_orbit}")
print(f"  gsd_m:          {s1c_scene.gsd_m}")
print()
print(f"Normalised name: {_normalise_satellite_name(s1c_scene.satellite)}")

print()
print("SAR processing pipeline for flood mapping:")
flood_pl = (
    client.pipeline("flood-map")
    .despeckle(filter="enhanced_lee", window=7)   # reduce speckle
    .calibrate(output_type="sigma0", in_db=True)  # DN -> sigma0 dB
    .flood_map(threshold=-15.0)                   # threshold-based water mask
    .cog(compress="deflate")                       # export COG
)
print(f"  Steps: {[s.step_type for s in flood_pl._steps]}")

# CLI equivalents:
print()
print("CLI:")
print("  PyGeoFetch sar despeckle s1c_vv.tif --filter enhanced_lee --window 7")
print("  PyGeoFetch sar calibrate s1c_vv.tif --output-type sigma0 --db")
print("  PyGeoFetch sar flood-map s1c_post.tif --threshold -15.0")

Sentinel-1C scene:
  satellite:      SENTINEL-1C
  product_type:   GRD
  polarisation:   VV+VH
  pass_direction: ascending
  relative_orbit: 37
  gsd_m:          10.0

Normalised name: S1C

SAR processing pipeline for flood mapping:
  Steps: ['despeckle', 'calibrate', 'flood_map', 'cog']

CLI:
  PyGeoFetch sar despeckle s1c_vv.tif --filter enhanced_lee --window 7
  PyGeoFetch sar calibrate s1c_vv.tif --output-type sigma0 --db
  PyGeoFetch sar flood-map s1c_post.tif --threshold -15.0


## Workflow 3: Burn Severity — dNBR

In [6]:
# dNBR = pre-fire NBR - post-fire NBR
# Requires Bands: B08 (NIR), B12 (SWIR2)
# Thresholds: >0.66 = high severity, 0.27-0.66 = moderate, <0.27 = low

print("dNBR workflow:")
print()
print("1. Search pre-fire and post-fire scenes")
print("   query_pre  = SearchQuery(bbox=..., end_date='2024-07-01')")
print("   query_post = SearchQuery(bbox=..., start_date='2024-07-15')")
print()
print("2. Download B08 + B12 for both epochs")
print("   options = DownloadOptions(bands=['B08','B12'])")
print()
print("3. Compute dNBR")
print("   dnbr = client.indices.dnbr(")
print("       pre_nir='B08_pre.tif', pre_swir2='B12_pre.tif',")
print("       post_nir='B08_post.tif', post_swir2='B12_post.tif')")
print()
print("4. Vectorize burn severity classes")
print("   vecs = client.post.vectorize(str(dnbr.output_path), threshold=0.27)")

# dNBR severity classification
severity = [
    ("> 0.66",    "High severity — stand-replacing fire"),
    ("0.44-0.66", "Moderate-high severity"),
    ("0.27-0.44", "Moderate-low severity"),
    ("0.10-0.27", "Low severity"),
    ("-0.25-0.10","Unburned / enhanced regrowth"),
    ("< -0.25",   "Post-fire regrowth"),
]
print()
print("dNBR severity thresholds (USGS/Key & Benson 2006):")
for rng, desc in severity:
    print(f"  {rng:<15}  {desc}")

dNBR workflow:

1. Search pre-fire and post-fire scenes
   query_pre  = SearchQuery(bbox=..., end_date='2024-07-01')
   query_post = SearchQuery(bbox=..., start_date='2024-07-15')

2. Download B08 + B12 for both epochs
   options = DownloadOptions(bands=['B08','B12'])

3. Compute dNBR
   dnbr = client.indices.dnbr(
       pre_nir='B08_pre.tif', pre_swir2='B12_pre.tif',
       post_nir='B08_post.tif', post_swir2='B12_post.tif')

4. Vectorize burn severity classes
   vecs = client.post.vectorize(str(dnbr.output_path), threshold=0.27)

dNBR severity thresholds (USGS/Key & Benson 2006):
  > 0.66           High severity — stand-replacing fire
  0.44-0.66        Moderate-high severity
  0.27-0.44        Moderate-low severity
  0.10-0.27        Low severity
  -0.25-0.10       Unburned / enhanced regrowth
  < -0.25          Post-fire regrowth


## Workflow 4: Precise Orbit Files for InSAR

In [7]:
# Capability 3: engine.fetch_orbit_file()
# InSAR (Interferometric SAR) requires precise orbit files (POEORB)
# published 21 days after acquisition by ESA

from pygeofetch.core.orbits import (
    _parse_acquisition_datetime, _parse_satellite,
    _orbit_covers_datetime, _find_cached_orbit
)
import tempfile
from pathlib import Path
from datetime import datetime

print("Orbit file management:")
print()

# Parse a real product name
product_name = "S1C_IW_SLC__1SDV_20260601T053000_20260601T053027_053442_067B1A_F3A2"
acq_dt = _parse_acquisition_datetime(product_name)
sat    = _parse_satellite(product_name)

print(f"Product:          {product_name[:55]}...")
print(f"Acquisition time: {acq_dt}")
print(f"Satellite:        {sat}")

# Check orbit file validity window
fname = "S1C_OPER_AUX_POEORB_OPOD_20260622T121000_V20260601T000000_20260603T000000.EOF"
covers = _orbit_covers_datetime(fname, acq_dt)
print()
print(f"Orbit file: {fname[:60]}...")
print(f"Covers acquisition: {covers}")

# Demo caching
with tempfile.TemporaryDirectory() as tmp:
    orbit_dir = Path(tmp)
    (orbit_dir / fname).write_text("<EOF>orbit data</EOF>")
    cached = _find_cached_orbit(orbit_dir, sat, acq_dt, "precise")
    print(f"Cache hit: {cached is not None} ({cached.name if cached else 'none'})")

print()
print("engine.fetch_orbit_file() usage:")
print("  # Fetch precise orbit file (21-day delay from acquisition)")
print("  path = engine.fetch_orbit_file(")
print("      product_name='S1C_IW_SLC__1SDV_20260601T053000...',")
print("      output_dir='./orbits/',")
print("      orbit_type='precise',  # or 'restituted' for near-real-time")
print("  )")

Orbit file management:

Product:          S1C_IW_SLC__1SDV_20260601T053000_20260601T053027_053442...
Acquisition time: 2026-06-01 05:30:00
Satellite:        S1C

Orbit file: S1C_OPER_AUX_POEORB_OPOD_20260622T121000_V20260601T000000_20...
Covers acquisition: True
Cache hit: True (S1C_OPER_AUX_POEORB_OPOD_20260622T121000_V20260601T000000_20260603T000000.EOF)

engine.fetch_orbit_file() usage:
  # Fetch precise orbit file (21-day delay from acquisition)
  path = engine.fetch_orbit_file(
      product_name='S1C_IW_SLC__1SDV_20260601T053000...',
      output_dir='./orbits/',
      orbit_type='precise',  # or 'restituted' for near-real-time
  )


## Workflow 5: Automated Weekly Monitoring

In [ ]:
# Full automated monitoring function with new capabilities

def run_monitoring_workflow(
    name: str,
    bbox: str,
    output_dir: Path,
    providers: list = None,
    cloud_max: int = 15,
    bands: list = None,
):
    output_dir.mkdir(parents=True, exist_ok=True)
    providers = providers or ["aws_earth", "planetary_computer"]
    bands     = bands or ["B04", "B08"]

    # Search
    query = SearchQuery(
        bbox=BoundingBox.from_string(bbox),
        start_date="2024-01-01",
        end_date="2024-06-01",
        cloud_cover_max=cloud_max,
        max_results=20,
        sort_by="cloud_cover",
        sort_ascending=True,
    )
    results = client.search(query, providers=providers)

    # Download top 3
    to_download = results[:3]
    options = DownloadOptions(
        parallel=2, retry_attempts=3, resume=True,
        on_failure="skip", bands=bands,
    )
    dl = client.download(to_download, output_dir, options)

    # BUG 2 contract verified: always len(dl) == len(to_download)
    assert len(dl) == len(to_download)

    succeeded = [r for r in dl if r.success]
    return {
        "name":          name,
        "scenes_found":  len(results),
        "downloaded":    len(succeeded),
        "failed":        len(dl) - len(succeeded),
        "status":        "ok" if succeeded else "no_data",
    }

# Run the monitoring workflow
report = run_monitoring_workflow(
    name="NYC Sentinel-2 NDVI",
    bbox="-74.1,40.6,-73.7,40.9",
    output_dir=Path("output/monitoring/nyc"),
)

print("Monitoring report:")
for k, v in report.items():
    print(f"  {k:<20} {v}")

20:38:54 INFO [  searcher] Searching 2 provider(s): aws_earth, planetary_computer
20:38:56 INFO [planetary_computer] Planetary Computer: 20 results
20:38:56 INFO [  searcher]   planetary_computer: 20 results
20:38:57 INFO [  searcher]   aws_earth: 20 results
20:38:57 INFO [  searcher] Search complete: 20 unique results from 2 providers
20:38:57 INFO [downloader] Downloading 3 scene(s) to: output/monitoring/nyc
20:38:57 INFO [downloader] Downloading 'S2A_MSIL2A_20240426T153941_R011_T18TXK_20240427T001356' from 'planetary_computer' → output/monitoring/nyc/planetary_computer
20:38:57 INFO [downloader] Downloading 'S2B_18TXL_20240524_0_L2A' from 'aws_earth' → output/monitoring/nyc/aws_earth
20:38:57 WARN [downloader] Download attempt 1/4 failed for 'S2B_18TXL_20240524_0_L2A': name 'resolve_band_keys' is not defined. Retrying in 0.8s...
20:38:58 WARN [downloader] Download attempt 2/4 failed for 'S2B_18TXL_20240524_0_L2A': name 'resolve_band_keys' is not defined. Retrying in 1.6s...
20:38:58

---
## Summary
- Workflow 1: NDVI time series with best-per-month scene selection
- Workflow 2: SAR flood mapping using Sentinel-1C (new satellite, same API)
- Workflow 3: dNBR burn severity with pre/post fire NBR
- Workflow 4: Precise orbit file management for InSAR (Capability 3)
- Workflow 5: Automated monitoring with BUG 2 contract verification

**Next:** Notebook 07 — Copernicus & Authenticated Providers